In [ ]:
!pip install -q albumentations torchmetrics pytorch-lightning clearml python-dotenv

In [ ]:
import os
import math
import shutil
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback, TQDMProgressBar
from torch.utils.data import DataLoader, Dataset
from torchmetrics import JaccardIndex
from albumentations.pytorch import ToTensorV2
from clearml import Task
import cv2
import numpy as np
from google.colab import files
from dotenv import load_dotenv

In [ ]:
from google.colab import userdata, drive

os.environ['CLEARML_API_ACCESS_KEY'] = userdata.get('CLEARML_API_ACCESS_KEY')
os.environ['CLEARML_API_SECRET_KEY'] = userdata.get('CLEARML_API_SECRET_KEY')

In [ ]:
drive.mount('/content/drive')


source_path = '/content/drive/MyDrive/'

tasks = [
    {
        "files": ["leftImg8bit_trainvaltest.zip", "gtFine_trainvaltest.zip"],
        "dest": "/content/data/cityscapes/"
    }
]

for task in tasks:
    if not os.path.exists(task["dest"]):
        os.makedirs(task["dest"])

    for zip_file in task["files"]:
        full_path = os.path.join(source_path, zip_file)

        if os.path.exists(full_path):
            print(f"Unpacking {zip_file} in {task['dest']}...")
            !unzip -q -o "{full_path}" -d "{task['dest']}"
        else:
            print(f"Error, file not found: {full_path}")

In [ ]:
CONFIG = {
    "project_name": "Segmentation_Urban_Scene_CourseWork",
    "task_name": "DinoV2_Small_Cityscapes_E2_ClassBalancedFocal",
    "data_dir": "./data/cityscapes",
    "model_name": "dinov2_vits14",
    "classes": 19,
    "batch_size": 16,
    "lr": 1e-4,
    "weight_decay": 0.05,
    "epochs": 50,
    "image_size": (518, 1022),
    "freeze_backbone": False,
    "loss_name": "weighted_focal",
    "focal_gamma": 2.0,
    "focal_ce_balance": 0.5,
    "class_weights": None,  # auto-computed from Cityscapes train split
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

# Set precision for Tensor Cores
torch.set_float32_matmul_precision('medium')

# Set allocator to avoid fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

task = Task.init(
    project_name=CONFIG["project_name"],
    task_name=CONFIG["task_name"],
    output_uri=False
)
task.connect(CONFIG)

def compute_cityscapes_class_weights(data_dir, num_classes=19, c=1.02):
    """Compute ENet-style class weights from Cityscapes train split masks."""
    mapping = {
        7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
        23: 10, 24: 11, 25: 12, 26: 13, 27: 14, 28: 15, 31: 16, 32: 17, 33: 18
    }

    masks_root = Path(data_dir) / 'gtFine' / 'train'
    if not masks_root.exists():
        raise FileNotFoundError(f'Missing Cityscapes train masks: {masks_root}')

    counts = np.zeros(num_classes, dtype=np.float64)

    for city_dir in sorted(masks_root.iterdir()):
        if not city_dir.is_dir():
            continue
        for mask_path in city_dir.glob('*_gtFine_labelIds.png'):
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                continue
            for src_id, train_id in mapping.items():
                counts[train_id] += np.sum(mask == src_id)

    total = counts.sum()
    if total <= 0:
        raise RuntimeError('No valid labeled pixels found for class-weight computation.')

    probs = counts / total
    weights = 1.0 / np.log(c + probs)
    weights = (weights / weights.mean()).tolist()
    return [float(f'{w:.6f}') for w in weights]


In [ ]:
class CityscapesDataset(Dataset):
    def __init__(self, root_dir, split='train', augmentation=None):
        self.root_dir = root_dir
        self.images_dir = os.path.join(root_dir, 'leftImg8bit', split)
        self.masks_dir = os.path.join(root_dir, 'gtFine', split)
        self.augmentation = augmentation
        self.ids = []

        for city in os.listdir(self.images_dir):
            img_dir = os.path.join(self.images_dir, city)
            mask_dir = os.path.join(self.masks_dir, city)
            for file_name in os.listdir(img_dir):
                if file_name.endswith('_leftImg8bit.png'):
                   self.ids.append({
                       'image': os.path.join(img_dir, file_name),
                       'mask': os.path.join(mask_dir, file_name.replace('_leftImg8bit.png', '_gtFine_labelIds.png'))
                   })

        # Cityscapes to training labels mapping
        self.mapping = {
            7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
            23: 10, 24: 11, 25: 12, 26: 13, 27: 14, 28: 15, 31: 16, 32: 17, 33: 18
        }

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        sample = self.ids[i]
        image = cv2.imread(sample['image'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(sample['mask'], 0)

        # Map Cityscapes labels to training labels
        mask_mapped = np.ones_like(mask) * 255
        for k, v in self.mapping.items():
            mask_mapped[mask == k] = v
        mask = mask_mapped

        if self.augmentation:
            sample = self.augmentation(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']

        return image, mask.long()

In [ ]:
class RoadDataModule(pl.LightningDataModule):
  def __init__(self, data_dir, batch_size, image_size):
      super().__init__()
      self.data_dir = data_dir
      self.batch_size = batch_size
      self.image_size = image_size

  def setup(self, stage=None):
      # Training augmentations with resize to 518x1022
      # More aggressive augmentations for fine-tuning
      self.train_transform = A.Compose([
          A.Resize(height=518, width=1022),
          A.HorizontalFlip(p=0.5),
          A.OneOf([
              A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
              A.RandomGamma(gamma_limit=(70, 120), p=1.0),
              A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35, p=1.0),
              A.RandomShadow(shadow_roi=(0.0, 0.4, 1.0, 1.0), num_shadows_lower=1, num_shadows_upper=2, shadow_dimension=5, p=1.0),
          ], p=0.75),
          A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.08, p=0.4),
          A.OneOf([
              A.RandomFog(fog_coef_lower=0.08, fog_coef_upper=0.3, alpha_coef=0.08, p=1.0),
              A.RandomRain(blur_value=3, brightness_coefficient=0.9, p=1.0),
              A.RandomSnow(snow_point_lower=0.12, snow_point_upper=0.35, brightness_coeff=1.8, p=1.0),
          ], p=0.35),
          A.OneOf([
              A.GaussNoise(var_limit=(10.0, 60.0), p=1.0),
              A.GaussianBlur(blur_limit=(3, 7), p=1.0),
              A.MotionBlur(blur_limit=5, p=1.0),
          ], p=0.25),
          A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
          ToTensorV2()
      ])

      # Validation augmentations with resize to 518x1022
      self.valid_transform = A.Compose([
          A.Resize(height=518, width=1022),
          A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
          ToTensorV2()
      ])

      self.train_ds = CityscapesDataset(
          self.data_dir,
          split='train',
          augmentation=self.train_transform
      )
      self.val_ds = CityscapesDataset(
          self.data_dir,
          split='val',
          augmentation=self.valid_transform
      )

  def train_dataloader(self):
      return DataLoader(
          self.train_ds,
          batch_size=self.batch_size,
          shuffle=True,
          num_workers=2,
          pin_memory=True
      )

  def val_dataloader(self):
      return DataLoader(
          self.val_ds,
          batch_size=self.batch_size,
          shuffle=False,
          num_workers=2,
          pin_memory=True
      )

In [ ]:
class LinearSegmentationHead(nn.Module):
    """
    Linear segmentation head for DINOv2.
    Handles patch-based features from DINOv2 (14x14 patches) and upsamples to full resolution.
    """
    def __init__(self, embed_dim=384, num_classes=19, patch_size=14):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_classes = num_classes
        self.patch_size = patch_size

        # Simple linear classifier head
        self.linear = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, x, h, w):
        """
        Args:
            x: patch features from DINOv2, shape (B, N, C) where N = num_patches
            h, w: target output height and width
        Returns:
            logits: shape (B, num_classes, h, w)
        """
        B, N, C = x.shape

        # Calculate patch grid dimensions
        # For 518x1022 image with patch_size=14: 37 x 73 patches
        patch_h = h // self.patch_size
        patch_w = w // self.patch_size

        # Reshape to spatial dimensions: (B, C, patch_h, patch_w)
        x = x.reshape(B, patch_h, patch_w, C).permute(0, 3, 1, 2)

        # Apply linear classifier
        x = self.linear(x)

        # Upsample to original resolution using bilinear interpolation
        x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

        return x


class ClearMLMetricsCallback(Callback):
    def __init__(self, task):
        super().__init__()
        self._logger = task.get_logger()

    def _log_metrics(self, trainer, stage):
        for name, value in trainer.callback_metrics.items():
            if isinstance(value, torch.Tensor):
                value = value.detach().cpu().item()
            if isinstance(value, (int, float)) and math.isfinite(value):
                self._logger.report_scalar(stage, name, value, iteration=trainer.current_epoch)

    def on_train_epoch_end(self, trainer, pl_module):
        self._log_metrics(trainer, "train")

    def on_validation_epoch_end(self, trainer, pl_module):
        self._log_metrics(trainer, "val")


class RoadModel(pl.LightningModule):
    def __init__(self, model_name='dinov2_vits14', num_classes=19, lr=1e-4,
                 weight_decay=0.05, epochs=50, freeze_backbone=True,
                 loss_name='weighted_focal', focal_gamma=2.0, focal_ce_balance=0.5, class_weights=None):
        super().__init__()
        self.save_hyperparameters()

        # Load DINOv2 small model from torch hub
        self.backbone = torch.hub.load('facebookresearch/dinov2', model_name)
        self.embed_dim = 384  # DINOv2 small embedding dimension
        self.patch_size = 14  # DINOv2 patch size

        # Freeze backbone if requested
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            print("Backbone frozen!")

        # Add segmentation head
        self.segmentation_head = LinearSegmentationHead(
            embed_dim=self.embed_dim,
            num_classes=num_classes,
            patch_size=self.patch_size
        )

        # Loss function configuration for E2 (class-balanced + focal)
        self.loss_name = loss_name
        self.focal_gamma = focal_gamma
        self.focal_ce_balance = focal_ce_balance
        if class_weights is not None:
            cw = torch.tensor(class_weights, dtype=torch.float32)
            self.register_buffer('class_weights', cw)
        else:
            self.class_weights = None

        # Metrics
        self.train_miou = JaccardIndex(task="multiclass", num_classes=num_classes, ignore_index=255)
        self.val_miou = JaccardIndex(task="multiclass", num_classes=num_classes, ignore_index=255)

    def forward(self, x):
        B, C, H, W = x.shape

        # Get patch features from DINOv2
        # DINOv2 returns features for each patch (removes CLS token)
        features = self.backbone.forward_features(x)

        # Remove CLS token (first token) and keep only patch tokens
        patch_features = features['x_norm_patchtokens']  # (B, N, embed_dim)

        # Pass through segmentation head
        logits = self.segmentation_head(patch_features, H, W)

        return logits

    def compute_loss(self, logits, masks):
        ce_loss = F.cross_entropy(
            logits,
            masks,
            ignore_index=255,
            weight=self.class_weights if hasattr(self, 'class_weights') else None
        )

        if self.loss_name in ('focal', 'weighted_focal'):
            probs = torch.softmax(logits, dim=1)
            targets = masks.clone()
            valid = targets != 255
            if valid.sum() == 0:
                return ce_loss

            # Prevent ignore_index (255) from being used as gather indices.
            targets_safe = targets.clone()
            targets_safe[~valid] = 0
            targets_safe = targets_safe.clamp(min=0, max=logits.shape[1] - 1)

            pt = probs.gather(1, targets_safe.unsqueeze(1)).squeeze(1)
            pt_valid = pt[valid].clamp(1e-6, 1 - 1e-6)
            focal_loss = (-(1.0 - pt_valid) ** self.focal_gamma * torch.log(pt_valid)).mean()
            return self.focal_ce_balance * ce_loss + (1.0 - self.focal_ce_balance) * focal_loss

        return ce_loss

    def training_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)

        loss = self.compute_loss(logits, masks)

        # Calculate mIoU
        preds = torch.argmax(logits, dim=1)
        miou = self.train_miou(preds, masks)

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log("train_miou", miou, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        return loss

    def validation_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)

        loss = self.compute_loss(logits, masks)

        # Calculate mIoU
        preds = torch.argmax(logits, dim=1)
        miou = self.val_miou(preds, masks)

        self.log("val_loss", loss, on_epoch=True, prog_bar=True, logger=True)
        self.log("val_miou", miou, on_epoch=True, prog_bar=True, logger=True)

        return loss

    def configure_optimizers(self):
        # Only optimize segmentation head if backbone is frozen
        if self.hparams.freeze_backbone:
            params = self.segmentation_head.parameters()
        else:
            # Use different learning rates for backbone and head
            params = [
                {"params": self.segmentation_head.parameters(), "lr": self.hparams.lr},
                {"params": self.backbone.parameters(), "lr": self.hparams.lr * 0.1}
            ]

        optimizer = torch.optim.AdamW(
            params,
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
            betas=(0.9, 0.999),
            eps=1e-8
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.hparams.epochs,
            eta_min=1e-6
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch"
            }
        }

In [ ]:

if CONFIG["class_weights"] is None:
    CONFIG["class_weights"] = compute_cityscapes_class_weights(
        CONFIG["data_dir"],
        num_classes=CONFIG["classes"]
    )
    print("Computed class weights from Cityscapes train split:")
    print(CONFIG["class_weights"])
    task.connect({"computed_class_weights": CONFIG["class_weights"]})

dm = RoadDataModule(
    CONFIG["data_dir"],
    CONFIG["batch_size"],
    CONFIG["image_size"]
)

model = RoadModel(
    model_name=CONFIG["model_name"],
    num_classes=CONFIG["classes"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    epochs=CONFIG["epochs"],
    freeze_backbone=CONFIG["freeze_backbone"],
    loss_name=CONFIG["loss_name"],
    focal_gamma=CONFIG["focal_gamma"],
    focal_ce_balance=CONFIG["focal_ce_balance"],
    class_weights=CONFIG["class_weights"]
)

checkpoint_callback = pl.callbacks.ModelCheckpoint(
    monitor="val_miou",
    dirpath="./checkpoints",
    filename="dinov2-small-cityscapes-{epoch:02d}-{val_miou:.4f}",
    save_top_k=1,
    mode="max",
)

trainer = pl.Trainer(
    max_epochs=CONFIG["epochs"],
    accelerator="gpu",
    devices=1,
    logger=True,
    callbacks=[
        checkpoint_callback,
        TQDMProgressBar(refresh_rate=20),
        ClearMLMetricsCallback(task)
    ],
    precision="16-mixed",
    accumulate_grad_batches=4
)

trainer.fit(model, datamodule=dm)

In [ ]:
best_model_path = trainer.checkpoint_callback.best_model_path

best_model = RoadModel.load_from_checkpoint(best_model_path)
best_model.to(CONFIG["device"])
best_model.eval()

# Validate the best model
val_results = trainer.validate(best_model, datamodule=dm)[0]

metrics_data = {
    "Metric": ["Loss", "mIoU", "Percent"],
    "Value": [
        f"{val_results['val_loss']:.4f}",
        f"{val_results['val_miou']:.4f}",
        f"{val_results['val_miou']*100:.2f}%"
    ]
}
df = pd.DataFrame(metrics_data)

print("\n" + "="*30)
print("Results:" + "\n")
print("="*30)
display(df)

def visualize_predictions(model, datamodule, num_samples=3):
    model.to(CONFIG["device"])
    model.eval()

    val_loader = datamodule.val_dataloader()
    images, masks = next(iter(val_loader))

    images = images.to(CONFIG["device"])
    with torch.no_grad():
        logits = model(images)
        preds = torch.argmax(logits, dim=1)

    images = images.cpu()
    masks = masks.cpu()
    preds = preds.cpu()

    plt.figure(figsize=(18, num_samples * 4))

    for i in range(min(num_samples, len(images))):
        # Denormalize image for visualization
        img = images[i].permute(1, 2, 0).numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)

        plt.subplot(num_samples, 3, i*3 + 1)
        plt.imshow(img)
        plt.title("Original")
        plt.axis("off")

        plt.subplot(num_samples, 3, i*3 + 2)
        plt.imshow(masks[i])
        plt.title("Ground truth")
        plt.axis("off")

        plt.subplot(num_samples, 3, i*3 + 3)
        plt.imshow(preds[i])
        plt.title(f"Prediction")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

visualize_predictions(best_model, dm)

In [ ]:
best_model_path = trainer.checkpoint_callback.best_model_path

model_name = os.path.basename(best_model_path)

current_task = Task.get_task(task_id=task.id)

current_task.update_output_model(
    model_path=best_model_path,
    model_name=model_name,
    iteration=trainer.current_epoch
)

In [ ]:
task.close()